# Leveraged ETFs on real data: matched windows and rolling-window evidence

Empirical companion to the article [*Why 3× Daily Leverage Is Not 3× Long-Term Return*](https://dvirross.com/quant-research/notes/why-3x-daily-leverage-is-not-3x-long-term-return). Every market-facing figure in the article (Figures 2 to 5) is produced here from **observed** daily history. No simulated return path appears in this notebook. The mathematical model and Monte Carlo experiments live separately in [`leveraged-etf-model-simulations.ipynb`](leveraged-etf-model-simulations.ipynb).

**Questions**

1. Can a real fall-and-recovery in QQQ be seen in the real TQQQ and SQQQ prices? (Figure 2)
2. Do real QQQ windows with the *same* length, the *same* endpoint return and the *same* number of positive days still give different idealized 3× outcomes? (Figure 3)
3. Across all rolling windows in five years of history, how large is the compounding shortfall, and what explains it? (Figure 4)
4. Do more positive days push the idealized 3× outcome up, and are green days enough to determine it? (Figure 5)

**Idealized 3× product.** The model applies `value_t = value_{t-1} × (1 + 3 r_t)` to QQQ's own daily returns `r_t`. It ignores fees, financing, tracking error and distributions. It is a model of the daily-reset mechanism, not a replica of TQQQ. The code is in [`analysis/empirical.py`](../analysis/empirical.py); the Plotly builders are in [`analysis/figures.py`](../analysis/figures.py).

## 1. Data and conventions

Files: `data/ibkr-qqq-daily-5y.csv`, `data/ibkr-tqqq-daily-5y.csv`, `data/ibkr-sqqq-daily-5y.csv` (see `data/README.md`).

* **Source:** Interactive Brokers historical price history, retrieved on 2026-10-07.
* **Bars:** daily, regular trading hours only (`outside_rth=false`), chart source `Last`, corporate-action metadata requested.
* **Price convention:** observed daily bar closes, **not** total-return adjusted. Distributions are not added back. Split adjustments, if any, are as returned by the source; all analysis uses ratios within a window, so level conventions do not matter.
* **Benchmark:** QQQ is a tradable ETF on the Nasdaq-100. TQQQ and SQQQ target the **index**, not QQQ shares, so QQQ is a proxy.
* **Missing data:** none. The loader raises an error on gaps, duplicate dates or non-positive prices, and requires the three files to cover identical dates.
* **Windows:** a window of `H` returns starts from a *base close* and ends at the close `H` trading days later. Rolling windows overlap and are therefore strongly dependent; the number of independent windows is roughly the sample length divided by `H`.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
from IPython.display import display

_here = Path.cwd().resolve()
REPO_ROOT = _here.parent if _here.name == "notebooks" else _here
sys.path.insert(0, str(REPO_ROOT))
from analysis import empirical as emp, figures as figs
from analysis.run import MATCH_HORIZON, ROLL_HORIZON, MATCH_TOLERANCE, MATCH_MIN_RETURN, horizon_table

FIG_DIR, RESULTS = REPO_ROOT / "figures", REPO_ROOT / "results"
EXPORT = True
MANIFEST = {}

close = emp.load_aligned(REPO_ROOT / "data")
print(f"{len(close)} daily closes per fund, {close.index.min().date()} to {close.index.max().date()}; identical dates for QQQ, TQQQ, SQQQ")
display(close.describe().loc[["min", "max"]].round(2))

ret = close.pct_change().dropna()
print("Largest one-day moves in QQQ:", f"{ret.QQQ.min():+.2%} ({ret.QQQ.idxmin().date()}) and {ret.QQQ.max():+.2%} ({ret.QQQ.idxmax().date()})")
beta = pd.Series({k: np.polyfit(ret.QQQ, ret[k], 1)[0] for k in ("TQQQ", "SQQQ")})
print("Daily-return regression slope on QQQ (should be near +3 and -3):", beta.round(2).to_dict())

1253 daily closes per fund, 2021-10-11 to 2026-10-07; identical dates for QQQ, TQQQ, SQQQ


,QQQ,TQQQ,SQQQ
min,260.10,8.08,31.83
max,759.66,87.22,1652.25


Largest one-day moves in QQQ: -6.21% (2025-04-04) and +12.00% (2025-04-09)


Daily-return regression slope on QQQ (should be near +3 and -3): {'TQQQ': 2.97, 'SQQQ': -2.97}


In [2]:
def save(fig, name, alt, caption, assumptions, size):
    '''Export the Plotly JSON spec and a static PNG fallback (PNG needs kaleido and a local Chrome/Chromium).'''
    MANIFEST[name] = {"spec": f"figures/{name}.json", "static": f"figures/{name}.png", "alt": alt, "caption": caption, "assumptions": assumptions}
    if EXPORT:
        figs.export(fig, FIG_DIR, name, size)

## 2. A real fall and recovery in two sessions (article Figure 2)

**Search rule (fixed before looking at results).** Consider every two-session window (three consecutive closes). Keep those in which QQQ's two-day return is within ±0.10% of zero and the first session is a fall. Rank by the size of the first-day fall and take the largest. This is a selected illustration of a mechanism, not a statement about how often such a pattern occurs.

In [3]:
rt = emp.round_trip_search(close, tolerance=0.001, first_day_fall=True)
print(f"{len(rt)} qualifying windows; the largest first-day fall is the first row:")
display(rt.head(5).style.format({c: "{:+.4%}" for c in rt.columns if c not in ("base_date", "mid_date", "end_date")}))
best = rt.iloc[0]
win = close.loc[best.base_date:best.end_date]
display(win.round(2))
print(f"QQQ: {best.r1:+.2%} then {best.r2:+.2%} = {best.qqq_return:+.2%}")
print(f"Idealized +3x: {best.ideal_plus3:+.2%} | actual TQQQ: {best.tqqq_return:+.2%}")
print(f"Idealized -3x: {best.ideal_minus3:+.2%} | actual SQQQ: {best.sqqq_return:+.2%}")
fig2 = figs.fig_round_trip(close, best)
fig2.show()
save(fig2, "02-real-two-day-roundtrip",
     alt="Line chart of QQQ, TQQQ, SQQQ and an idealized +3× series normalized to 100 on 17 April 2025, for 17, 21 and 22 April. QQQ dips to 97.52 and finishes at 100.09; TQQQ falls to 92.69 and finishes at 99.78; SQQQ rises to 107.38 and finishes at 99.07.",
     caption="A real two-session fall and recovery. QQQ fell and then rose back to just above its starting close, yet both the daily +3× product TQQQ and the daily −3× product SQQQ finished below their starting values.",
     assumptions="Observed IBKR daily closes (regular trading hours), not total-return adjusted. QQQ is a proxy for the Nasdaq-100. Selected by the documented rule in this notebook; one illustration, not evidence about frequency. Actual fund returns include fees, financing and tracking effects that three closes cannot separate.",
     size=(1100, 520))

21 qualifying windows; the largest first-day fall is the first row:


,base_date,mid_date,end_date,r1,r2,qqq_return,ideal_plus3,ideal_minus3,tqqq_return,sqqq_return
0,2025-04-17 00:00:00,2025-04-21 00:00:00,2025-04-22 00:00:00,-2.4747%,+2.6252%,+0.0856%,-0.1331%,-1.0363%,-0.2214%,-0.9317%
1,2025-02-06 00:00:00,2025-02-07 00:00:00,2025-02-10 00:00:00,-1.2613%,+1.2105%,-0.0661%,-0.2899%,+0.0150%,-0.4302%,+0.2514%
2,2024-07-24 00:00:00,2024-07-25 00:00:00,2024-07-26 00:00:00,-1.1028%,+1.0256%,-0.0885%,-0.3333%,+0.1297%,-0.5653%,+0.4415%
3,2025-08-29 00:00:00,2025-09-02 00:00:00,2025-09-03 00:00:00,-0.8380%,+0.7867%,-0.0579%,-0.2131%,+0.0944%,-0.2798%,+0.2228%
4,2023-02-28 00:00:00,2023-03-01 00:00:00,2023-03-02 00:00:00,-0.8039%,+0.8276%,+0.0170%,+0.0112%,-0.1309%,-0.0904%,+0.0000%


,QQQ,TQQQ,SQQQ
date,,,
2025-04-17,444.10,22.58,193.20
2025-04-21,433.11,20.93,207.45
2025-04-22,444.48,22.53,191.40


QQQ: -2.47% then +2.63% = +0.09%
Idealized +3x: -0.13% | actual TQQQ: -0.22%
Idealized -3x: -1.04% | actual SQQQ: -0.93%


## 3. Matched windows: same endpoint, same green days, different leveraged outcomes (article Figure 3)

**Search rule (fixed before looking at results).**

1. Horizon `H = 20` daily returns (about one trading month); build every rolling window.
2. Keep windows whose QQQ cumulative return is at least +5%.
3. Consider sets of three windows that **do not overlap**, have the **same number of positive-return days**, and whose QQQ cumulative returns all lie within a band of width **0.5 percentage points**.
4. Among all such sets, take the one with the largest range of idealized 3× return; ties on the extreme windows are broken by the largest minimum gap between the three 3× outcomes.

The selection looks for a visible spread on purpose, because the aim is to illustrate a mechanism. It is not a random draw. The matching tolerance and all alternatives are listed in `results/matched_windows_20d_top_candidates.csv`.

In the idealized model, reordering the *same* daily returns would not change terminal wealth, because multiplication is commutative. The three windows below contain *different* return sequences that happen to share a length, an endpoint and a positive-day count.

In [4]:
w20 = emp.rolling_windows(close["QQQ"], MATCH_HORIZON, compare=close["TQQQ"])
m = emp.match_windows(w20, MATCH_HORIZON, MATCH_TOLERANCE, MATCH_MIN_RETURN)
print(f"{len(w20)} rolling {MATCH_HORIZON}-day windows; {m.n_candidates} qualifying sets of three; selected set below")
cols = ["base_date", "end_date", "bench_return", "naive_return", "lev_return", "deviation", "actual_return", "vol", "pos_days", "bench_max_drawdown"]
display(m.windows[cols].style.format({c: "{:+.2%}" for c in cols if c not in ("base_date", "end_date", "pos_days")} | {"base_date": "{:%Y-%m-%d}", "end_date": "{:%Y-%m-%d}"}))
q = close["QQQ"].pct_change()
for _, w in m.windows.iterrows():
    r = q.loc[w.base_date:w.end_date].iloc[1:]
    print(f"{w.base_date:%Y-%m-%d} to {w.end_date:%Y-%m-%d}: largest up day {r.max():+.2%}, largest down day {r.min():+.2%}, daily std {r.std():.2%}")

# sensitivity of the selection to the tolerance
for tol in (0.0025, 0.005, 0.0075, 0.01):
    mm = emp.match_windows(w20, MATCH_HORIZON, tol, MATCH_MIN_RETURN)
    print(f"tolerance {tol:.2%}: {mm.n_candidates:5d} candidate sets; 3x range {mm.windows.lev_return.max() - mm.windows.lev_return.min():.2%}")

fig3 = figs.fig_matched_windows(close["QQQ"], m.windows, MATCH_HORIZON)
fig3.show()
save(fig3, "03-matched-windows",
     alt="Two stacked charts over 20 trading days for three real QQQ windows that each rose about 8.5 to 9 percent with 12 positive days. Top: QQQ paths, one calm, one moderate and one very volatile with a drop of about 8 percent early on. Bottom: the idealized daily 3× product on the same days ends at about 126 for the window starting June 2022, about 128 for May 2023 and about 119 for April 2025, against a dashed line at about 126 for three times the QQQ gain.",
     caption="Three real 20-day QQQ windows with almost the same cumulative return (about +8.5% to +9%) and the same number of positive days (12 of 20). The idealized daily 3× product ends between about +19% and +28%, because the day-to-day path differs.",
     assumptions="Observed IBKR daily closes, not total-return adjusted. Windows found by the systematic rule in this notebook (largest 3× spread among non-overlapping windows with equal positive-day count and QQQ returns within 0.5 percentage points); selected to illustrate a mechanism, not randomly sampled. The 3× series is an idealized model of QQQ's own daily returns without fees, financing or tracking error.",
     size=(1100, 760))

1233 rolling 20-day windows; 1654 qualifying sets of three; selected set below


,base_date,end_date,bench_return,naive_return,lev_return,deviation,actual_return,vol,pos_days,bench_max_drawdown
0,2022-06-17,2022-07-19,+8.60%,+25.79%,+25.90%,+0.12%,+25.54%,+26.92%,12,-4.86%
1,2023-05-02,2023-05-31,+8.96%,+26.87%,+28.35%,+1.47%,+26.73%,+17.61%,12,-1.77%
2,2025-04-03,2025-05-02,+8.47%,+25.41%,+19.38%,-6.03%,+18.15%,+56.92%,12,-7.68%


2022-06-17 to 2022-07-19: largest up day +3.43%, largest down day -3.05%, daily std 1.70%
2023-05-02 to 2023-05-31: largest up day +2.56%, largest down day -1.27%, daily std 1.11%
2025-04-03 to 2025-05-02: largest up day +12.00%, largest down day -6.21%, daily std 3.59%


tolerance 0.25%:   344 candidate sets; 3x range 7.86%


tolerance 0.50%:  1654 candidate sets; 3x range 8.97%


tolerance 0.75%:  3610 candidate sets; 3x range 9.61%


tolerance 1.00%:  5843 candidate sets; 3x range 10.05%


## 4. Rolling windows: how large is the compounding shortfall? (article Figure 4)

**Choosing the horizon.** The five-year history holds about 1,250 trading days. Longer windows overlap more and give fewer independent observations. The table shows the sample sizes and basic statistics; **63 trading days (about one quarter)** is long enough for compounding to matter and gives 1,190 overlapping windows (about 19 non-overlapping quarters). Treat the scatter as a description of this one five-year episode, not as 1,190 independent experiments.

For each window the notebook records the QQQ cumulative return `R`, the idealized 3× return, `3R`, the deviation (3× return minus `3R`), annualized realized volatility, the number of positive days, the benchmark's maximum drawdown and the actual TQQQ return over the same dates.

The **shortfall relative to the smooth-path value** is `(1 + 3× return) / (1 + R)³ − 1`. It removes the effect of the trend and isolates the penalty from day-to-day variation. For small daily moves it is close to `exp(−3 Σ r²) − 1`, and `Σ r²` is approximately `σ² T` for annualized volatility `σ` and horizon `T` in years.

In [5]:
display(horizon_table(close).style.format({c: "{:.3f}" for c in ["share_lev_below_zero", "share_lev_above_naive", "median_drag", "corr_vol_drag", "spearman_posshare_levreturn", "rms_actual_tqqq_minus_ideal"]}))

w = emp.rolling_windows(close["QQQ"], ROLL_HORIZON, compare=close["TQQQ"])
print(f"\n{len(w)} overlapping {ROLL_HORIZON}-day windows; realized volatility {w.vol.min():.1%} to {w.vol.max():.1%}; QQQ return {w.bench_return.min():+.1%} to {w.bench_return.max():+.1%}")
quart = w.groupby(pd.qcut(w.vol, 4), observed=True).agg(n=("vol", "size"), median_vol=("vol", "median"), median_shortfall=("drag", "median"),
                                                        worst=("drag", "min"), mildest=("drag", "max"), median_deviation=("deviation", "median"))
display(quart.style.format({"median_vol": "{:.1%}", "median_shortfall": "{:.2%}", "worst": "{:.2%}", "mildest": "{:.2%}", "median_deviation": "{:+.2%}"}))
print(f"Correlation of realized volatility with the shortfall: {w[['vol', 'drag']].corr().iloc[0, 1]:.3f}")
up = w[w.bench_return > 0]
print(f"Windows where QQQ rose: {len(up)}; the idealized 3x return beat 3x the QQQ return in {(up.deviation > 0).sum()} of them ({(up.deviation > 0).mean():.0%}); it ended below zero in {(up.lev_return < 0).sum()} (QQQ gains of at most {up[up.lev_return < 0].bench_return.max():.2%})")
print(f"Median gap, actual TQQQ minus idealized 3x over the same windows: {(w.actual_return - w.lev_return).median():+.2%} (fees, financing, tracking and distributions are not separated here)")

fig4 = figs.fig_vol_drag(w, ROLL_HORIZON)
fig4.show()
save(fig4, "04-volatility-drag-rolling",
     alt="Scatter chart of 1,190 overlapping 63-day QQQ windows. Horizontal axis: realized volatility from about 11 to 39 percent. Vertical axis: the idealized 3× wealth relative to the cube of QQQ's growth, falling from about −1 percent at low volatility to about −10 percent at the highest volatility, closely following a dashed second-order model curve. Points are colored by QQQ's return over the window. A legend toggle shows the actual TQQQ results.",
     caption="Across 1,190 overlapping 63-day QQQ windows, the idealized daily 3× product falls further below the smooth-path value as realized volatility rises, closely following the second-order approximation. Hover a point for its dates and returns; use the legend to add actual TQQQ.",
     assumptions="Observed IBKR daily closes (QQQ, TQQQ), 2021-10-11 to 2026-10-07, not total-return adjusted. Overlapping windows are not independent (about 19 non-overlapping quarters). Shortfall = (1 + idealized 3× return) / (1 + QQQ return)³ − 1. The dashed curve is exp(−3σ²T) − 1 with T = 63/252. Actual TQQQ includes fees, financing and tracking effects.",
     size=(1100, 640))

,horizon_days,overlapping_windows,approx_independent_windows,share_lev_below_zero,windows_bench_up_lev_down,share_lev_above_naive,median_drag,corr_vol_drag,spearman_posshare_levreturn,rms_actual_tqqq_minus_ideal
0,20,1233,62,0.395,25,0.336,-0.008,-0.975,0.709,0.009
1,63,1190,19,0.363,38,0.342,-0.027,-0.992,0.762,0.031
2,126,1127,9,0.286,50,0.393,-0.056,-0.993,0.589,0.073
3,252,1001,4,0.158,15,0.472,-0.133,-0.994,0.418,0.194



1190 overlapping 63-day windows; realized volatility 10.6% to 39.3%; QQQ return -23.6% to +33.5%


,n,median_vol,median_shortfall,worst,mildest,median_deviation
vol,,,,,,
"(0.105, 0.164]",298,15.3%,-1.75%,-2.08%,-0.85%,+1.11%
"(0.164, 0.192]",297,17.2%,-2.23%,-2.77%,-1.98%,-1.27%
"(0.192, 0.279]",297,22.7%,-3.80%,-5.89%,-2.69%,-2.60%
"(0.279, 0.393]",298,33.0%,-7.78%,-10.55%,-5.69%,-4.18%


Correlation of realized volatility with the shortfall: -0.992
Windows where QQQ rose: 796; the idealized 3x return beat 3x the QQQ return in 325 of them (41%); it ended below zero in 38 (QQQ gains of at most 2.80%)
Median gap, actual TQQQ minus idealized 3x over the same windows: -3.00% (fees, financing, tracking and distributions are not separated here)


## 5. Positive days versus the leveraged outcome (article Figure 5)

If volatility drag does not make a loss inevitable, how much do positive days matter? The windows are grouped by their share of positive-return days. Hovering a point shows its dates, QQQ return, realized volatility and exact positive-day count.

In [6]:
bins = emp.bin_by_positive_share(w)
display(bins.style.format({"bench_median": "{:+.2%}", "lev_median": "{:+.2%}", "lev_min": "{:+.2%}", "lev_max": "{:+.2%}"}))
print(f"Spearman rank correlation, positive-day share vs idealized 3x return: {w[['pos_share', 'lev_return']].corr('spearman').iloc[0, 1]:.2f}")
print("Windows with exactly the same positive-day count still differ widely:")
display(w.groupby("pos_days").agg(n=("lev_return", "size"), qqq_min=("bench_return", "min"), qqq_max=("bench_return", "max"),
                                  lev_min=("lev_return", "min"), lev_median=("lev_return", "median"), lev_max=("lev_return", "max")).loc[34:42]
        .style.format({c: "{:+.1%}" for c in ["qqq_min", "qqq_max", "lev_min", "lev_median", "lev_max"]}))

fig5 = figs.fig_green_days(w, ROLL_HORIZON)
fig5.show()
save(fig5, "05-positive-days-rolling",
     alt="Box plots with every point shown, for 1,190 overlapping 63-day QQQ windows grouped by share of positive days, from 45 percent or fewer to more than 65 percent. The median idealized 3× return rises from about −33 percent to about +53 percent, but each group spans a wide range, for example from about −13 percent to +86 percent in the 55 to 60 percent group.",
     caption="More positive days push the idealized 3× outcome up, but the share of green days alone does not determine it: windows with similar positive-day shares end anywhere from a loss to a large gain, depending on the size of the daily moves.",
     assumptions="Observed IBKR daily closes of QQQ, 2021-10-11 to 2026-10-07, not total-return adjusted; 63-day overlapping windows (about 19 independent quarters). The 3× series is an idealized model without fees, financing or tracking error.",
     size=(1100, 560))

,positive_day_share,n,bench_median,lev_median,lev_min,lev_max
0,"(-0.001, 0.45]",160,-9.91%,-32.69%,-60.15%,+0.47%
1,"(0.45, 0.5]",180,-2.60%,-12.16%,-57.16%,+39.05%
2,"(0.5, 0.55]",275,+2.75%,+4.39%,-48.64%,+62.44%
3,"(0.55, 0.6]",272,+7.57%,+21.27%,-12.84%,+86.22%
4,"(0.6, 0.65]",242,+9.76%,+30.74%,-12.66%,+122.91%
5,"(0.65, 1.0]",61,+15.85%,+52.67%,+16.07%,+119.34%


Spearman rank correlation, positive-day share vs idealized 3x return: 0.76
Windows with exactly the same positive-day count still differ widely:


,n,qqq_min,qqq_max,lev_min,lev_median,lev_max
pos_days,,,,,,
34,83,-9.3%,+18.5%,-32.4%,+9.9%,+62.4%
35,83,-3.1%,+20.7%,-10.8%,+16.0%,+71.6%
36,98,-3.1%,+24.0%,-12.8%,+18.4%,+86.2%
37,91,-2.9%,+21.8%,-12.2%,+25.6%,+76.8%
38,87,-3.2%,+20.9%,-12.7%,+24.8%,+71.2%
39,92,-1.8%,+23.6%,-9.1%,+31.2%,+80.5%
40,63,+5.7%,+33.5%,+14.0%,+41.5%,+122.9%
41,38,+6.2%,+31.9%,+16.1%,+46.5%,+119.3%
42,17,+12.0%,+20.2%,+36.7%,+52.7%,+71.0%


## 6. What five years can and cannot say

* **One episode.** The sample covers 2021-10 to 2026-10: a 2022 drawdown, a long recovery and a sharp 2025 shock. Overlapping windows within it are not independent. A longer history (for example back to 2010, covering more rate and volatility regimes, or back to 1999–2003 and 2007–2009) would let the same code estimate how stable these relationships are; the loader accepts any file with the same columns.
* **Idealized versus actual.** The idealized series ignores fees, financing, tracking error and distributions. Actual TQQQ trails it by a median of about 3 percentage points per 63-day window here; this notebook does not decompose that gap.
* **Raw prices.** QQQ's distributions are not added back, so QQQ and idealized returns are slightly understated relative to total return.
* **Selection.** Figures 2 and 3 are selected to illustrate a mechanism. Figures 4 and 5 use every window, not a selection.
* **Not advice.** Nothing here is investment advice or a forecast.

In [7]:
if EXPORT:
    (FIG_DIR / "figures.json").write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    RESULTS.mkdir(exist_ok=True)
    rt.to_csv(RESULTS / "round_trip_candidates.csv", index=False)
    m.windows.to_csv(RESULTS / "matched_windows_20d.csv", index=False)
    m.runners_up.to_csv(RESULTS / "matched_windows_20d_top_candidates.csv", index=False)
    w.to_csv(RESULTS / f"rolling_{ROLL_HORIZON}d_windows.csv", index=False)
    bins.to_csv(RESULTS / f"positive_day_bins_{ROLL_HORIZON}d.csv", index=False)
    horizon_table(close).to_csv(RESULTS / "horizon_sensitivity.csv", index=False)
print("Exported figures:", ", ".join(MANIFEST))

Exported figures: 02-real-two-day-roundtrip, 03-matched-windows, 04-volatility-drag-rolling, 05-positive-days-rolling
